In [33]:
# Importations
import os
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import round,col, dayofmonth,month,year,quarter,when 
from pyspark.sql import functions as F

In [34]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [35]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

In [36]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerSilverTransformation") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

In [37]:
# Récupère la table weatherforectastapi_raw depuis PostgreSQL dans un DataFrame Spark
weatherforecastapi_raw_df = spark.read.jdbc(
    url=JDBC_URL,
    table='bronze.weatherforecastapi_raw',
    properties=JDBC_PROPS
)

In [38]:
# Nettoyage et enrichissement des données météo
# Transformation: arrondir les colonnes "wind_speed_100m", "wind_gusts_10m", "temperature_2m", "pressure_msl", "precipitation"
# Transformation: créer la colonne "day" à partir de "date" avec la méthode "dayofmonth"
# Transformation: créer la colonne "month" à partir de "date" avec la méthode "month"
# Transformation: créer la colonne "quarter" à partir de "date" avec la méthode "quarter"
# Transformation: créer la colonne "year" à partir de "date" avec la méthode "year"
# Transformation: formater la colonne "time" au format "HH:mm:ss"
# Transformation: créer les colonnes "hour_of_day", "minute_of_hour", "second_of_minute" à partir de "time"
# Transformation: créer la colonne "time_period" (Morning, Afternoon, Evening, Night) à partir de "hour_of_day"

df_transformed = (
    weatherforecastapi_raw_df
    .withColumn("wind_speed_100m", round(col("wind_speed_100m"), 2))
    .withColumn("wind_gusts_10m", round(col("wind_gusts_10m"), 2))
    .withColumn("temperature_2m", round(col("temperature_2m"), 2))
    .withColumn("pressure_msl", round(col("pressure_msl"), 2))
    .withColumn("precipitation", round(col("precipitation"), 2))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("year", year(col("date")))
    .withColumn("time", F.date_format(col("time"), "HH:mm:ss"))
    .withColumn("hour_of_day", F.hour(col("time")))
    .withColumn("minute_of_hour", F.minute(col("time")))
    .withColumn("second_of_minute", F.second(col("time")))
    .withColumn(
        "time_period",
        when((col("hour_of_day") >= 5) & (col("hour_of_day") < 12), "Morning")
        .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 17), "Afternoon")
        .when((col("hour_of_day") >= 17) & (col("hour_of_day") < 21), "Evening")
        .otherwise("Night")
    )
)

# Suppression des colonnes des métadonnées
df_transformed = df_transformed.drop("ingested_at","source_api")

df_transformed.show(5)

+----------+--------+---------+----------+--------+--------------------+---------------+--------------+--------------+------------+-------------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|      date|    time| latitude| longitude|  region|         region_name|wind_speed_100m|wind_gusts_10m|temperature_2m|pressure_msl|precipitation|day|month|quarter|year|hour_of_day|minute_of_hour|second_of_minute|time_period|
+----------+--------+---------+----------+--------+--------------------+---------------+--------------+--------------+------------+-------------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|2024-06-16|00:00:00|34.059753| -118.2375|Region A|Los Angeles, Cali...|           15.3|          29.9|          28.0|      1006.7|          0.0| 16|    6|      2|2024|          0|             0|               0|      Night|
|2024-06-16|00:00:00|36.801403|-119.44809|Region B|Fresno/Central Va...|           16.6|          31

In [ ]:
# Définition de la table silver.weatherforecastapi_cleaned avec les types de données appropriés
create_table_sql = """
CREATE TABLE IF NOT EXISTS silver.weatherforecastapi_cleaned (
    weather_id BIGSERIAL PRIMARY KEY,
    date DATE,
    time TIME,
    latitude NUMERIC(9,6),
    longitude NUMERIC(9,6),
    region VARCHAR(100),
    region_name VARCHAR(255),
    wind_speed_100m NUMERIC(6,2),
    wind_gusts_10m NUMERIC(6,2),
    temperature_2m NUMERIC(5,2),
    pressure_msl NUMERIC(7,2),
    precipitation NUMERIC(8,2),
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER,
    hour_of_day INTEGER,
    minute_of_hour INTEGER,
    second_of_minute INTEGER,
    time_period VARCHAR(20),
    CONSTRAINT unique_weather_record UNIQUE (date, time, region, region_name)
);
"""

# Connexion à PostgreSQL et exécution du script de création de table
with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        # Crée le schéma s'il n'existe pas
        cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
        
        # Crée la table avec script SQL
        cur.execute(create_table_sql)
        print("Schéma silver et table weatherforecastapi_cleaned créés avec les types corrects")
        
        # Ajouter la contrainte UNIQUE métier si elle n'existe pas
        try:
            cur.execute("""
                ALTER TABLE silver.weatherforecastapi_cleaned 
                ADD CONSTRAINT unique_weather_record UNIQUE (date, time, region, region_name);
            """)
            print("Contrainte UNIQUE ajoutée sur (date, time, region, region_name)")
        except (psycopg.errors.DuplicateTable, psycopg.errors.DuplicateObject):
            print("Contrainte UNIQUE existe déjà")
        

Schéma silver et table weatherforecastapi_cleaned créés avec les types corrects
Contrainte UNIQUE existe déjà


In [40]:
# Écriture dans silver.weatherforecastapi_cleaned avec protection anti-doublons

total_count = df_transformed.count()
print(f"Lignes à traiter: {total_count}")

# Filtrage des doublons (clé métier: date, time, region, region_name)
try:
    existing_df = spark.read.jdbc(url=JDBC_URL, table='silver.weatherforecastapi_cleaned', properties=JDBC_PROPS)
    
    existing_keys = existing_df.select("date", F.date_format(col("time"), "HH:mm:ss").alias("time"), "region", "region_name")
    df_to_insert = df_transformed.join(existing_keys, on=["date", "time", "region", "region_name"], how="left_anti")
    
    # Déduplication interne du lot sur la même clé métier
    df_to_insert = df_to_insert.dropDuplicates(["date", "time", "region", "region_name"])
    
    new_count = df_to_insert.count()
    print(f"Lignes existantes: {existing_df.count()} | Nouvelles: {new_count} | Doublons ignorés: {total_count - new_count}")
    
except Exception:
    df_to_insert = df_transformed.dropDuplicates(["date", "time", "region", "region_name"])
    new_count = df_to_insert.count()
    print(f"Table vide -> Insertion de {new_count} lignes après déduplication")

# Insertion triée par date, time, region
if new_count > 0:
    df_final = df_to_insert \
        .withColumn("time", F.to_timestamp(col("time"), "HH:mm:ss")) \
        .orderBy("date", "time", "region")
    
    df_final.write.jdbc(url=JDBC_URL, table="silver.weatherforecastapi_cleaned", mode="append", properties=JDBC_PROPS)
    print(f"{new_count} lignes insérées avec succès")
else:
    print("Aucune donnée à insérer")

# Vérification finale
final_count = spark.read.jdbc(url=JDBC_URL, table='silver.weatherforecastapi_cleaned', properties=JDBC_PROPS).count()
print(f"\nTotal en base: {final_count} lignes")

Lignes à traiter: 432
Lignes existantes: 432 | Nouvelles: 432 | Doublons ignorés: 0
432 lignes insérées avec succès

Total en base: 864 lignes
